In [4]:
# not with target but serial
import serial, time
import chipwhisperer as cw, time

scope = cw.scope()
scope.default_setup()

# pins must stay in serial mode so the CDC port is routed to them
scope.io.tio1 = "serial_tx"
scope.io.tio2 = "serial_rx"

ser = serial.Serial("/dev/ttyACM0", 115200, parity=serial.PARITY_EVEN,                    stopbits=1, bytesize=8, timeout=1)

# enter the bootloader: BOOT0 high, then reset
scope.io.pdic = "high"
scope.io.nrst = "low";  time.sleep(0.1)
scope.io.nrst = "high_z"; time.sleep(0.3)

ser.reset_input_buffer()          # drop any glitch bytes from the reset
ser.write(b"\x7f")
print("handshake:", ser.read(1).hex())          # expect 79

# Get ID: 0x02 + complement 0xFD
ser.write(bytes([0x02, 0xFD]))
print("get id:", ser.read(5).hex())             # expect 79 01 0435 79  # iF is NAK

scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_

In [5]:
import serial, time
import chipwhisperer as cw

scope = cw.scope()
scope.default_setup()
scope.io.tio1 = "serial_tx"
scope.io.tio2 = "serial_rx"

ser = serial.Serial("/dev/ttyACM0", 115200, parity=serial.PARITY_EVEN,
                    stopbits=1, bytesize=8, timeout=1)

def enter_bootloader():
    scope.io.pdic = "high"                 # BOOT0 high
    time.sleep(0.05)
    scope.io.nrst = "low";  time.sleep(0.2)
    scope.io.nrst = "high_z"; time.sleep(0.5)
    ser.reset_input_buffer()
    ser.write(b"\x7f")
    return ser.read(1)

def command(code):
    ser.reset_input_buffer()
    time.sleep(0.02)
    ser.write(bytes([code, code ^ 0xFF]))
    return ser.read(1)

for attempt in range(5):
    hs = enter_bootloader()
    print(attempt, "handshake:", hs.hex())
    if hs != b"\x79":
        continue
    time.sleep(0.05)
    ack = command(0x02)
    print("  get id ack:", ack.hex())
    if ack == b"\x79":
        n = ser.read(1)[0]
        print("  id:", ser.read(n + 1).hex(), ser.read(1).hex())
        break

scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_

In [10]:
import serial, time
import chipwhisperer as cw

scope = cw.scope()
scope.default_setup()
target = cw.target(scope, cw.targets.SimpleSerial)

scope.io.tio1 = "serial_tx"
scope.io.tio2 = "serial_rx"
target.ser.cwlite_usart.init(baud=115200, stopbits=1, parity="even")

ser = serial.Serial("/dev/ttyACM0", 115200, parity=serial.PARITY_EVEN,
                    stopbits=1, bytesize=8, timeout=1)

scope.io.pdic = "high"; time.sleep(0.05)
scope.io.nrst = "low";  time.sleep(0.2)
scope.io.nrst = "high_z"; time.sleep(0.5)
ser.reset_input_buffer()
ser.write(b"\x7f");           print("hs :", ser.read(1).hex())
ser.write(bytes([0x02, 0xFD])); print("id :", ser.read(5).hex())

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2       

SerialException: [Errno 2] could not open port /dev/ttyACM0: [Errno 2] No such file or directory: '/dev/ttyACM0'

In [3]:
def cmd(code):
    ser.write(bytes([code, code ^ 0xFF]))
    return ser.read(1)

# Get: bootloader version and supported commands
print(cmd(0x00).hex())          # 79
n = ser.read(1)[0]              # number of bytes that follow, minus 1
print(ser.read(n + 1).hex(), ser.read(1).hex())   # version, command list, ACK

# Read memory (0x11): address (4 bytes + XOR checksum), then length-1 + complement
def read_mem(addr, length):
    assert cmd(0x11) == b"\x79"
    a = addr.to_bytes(4, "big")
    ser.write(a + bytes([a[0] ^ a[1] ^ a[2] ^ a[3]]))
    if ser.read(1) != b"\x79":
        raise RuntimeError("address NACK (readout protection?)")
    ser.write(bytes([length - 1, (length - 1) ^ 0xFF]))
    if ser.read(1) != b"\x79":
        raise RuntimeError("length NACK")
    return ser.read(length)

print(read_mem(0x08000000, 16).hex())   # first 16 bytes of flash

IndexError: index out of range

In [12]:
%%bash
ls -l /dev/ttyACM*
dmesg | tail -20

crw-rw-r-- 1 root dialout 166, 1 Nov 16 15:41 /dev/ttyACM1


dmesg: read kernel buffer failed: Operation not permitted
